# Check an AI-written channel report

*Data Analysis in Python · Topic 14.2*

A colleague asked an AI assistant for Q3 2024's channel report, and its code runs without an error. Check its results against the data with the three questions, find its two silent errors, and write the report the way it should be.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the two cells that build the `sessions` table, and the AI's cell, which creates the `ai_report` table. Leave the AI's cell as it is.
2. In the first cell marked ✏️, check the totals. Create `revenue_gap`: the `ai_report` table's revenue total minus the `sessions` table's. Then create the `removed` table: the rows of the `sessions` table that the AI's filter dropped. Look at their sessions and revenue: what are they?
3. In the second cell marked ✏️, check the definitions. Create the `rate_gap` Series: for each channel group, the `ai_report` table's conversion rate minus its transactions divided by its sessions, with the channel groups as the index.
4. In the third cell marked ✏️, create the `report` table the way it should be: one row per channel group, with the columns `channel_group`, `sessions`, `transactions`, `revenue` and `conversion_rate`, sorted by revenue, highest first.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `revenue_gap` is \$62,637.66, and the `removed` table has the 530 refund rows the AI's filter dropped, with −\$62,637.66 of revenue. The `rate_gap` Series is largest for Email, at 0.0019. The `report` table has 9 channel groups, with 255,215 sessions and \$454,879.89 of revenue, and Email converts best, at 5.35%.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-07", "2024-08", "2024-09"]
START, END = "2024-07-01", "2024-09-30"  # Q3 2024

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}
raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## The AI's report

In [ ]:
# The AI's code. Run it, and leave it as it is: you'll check it.
# Prompt: "Write pandas code for Q3 2024's sessions, transactions, revenue and conversion rate by channel group."

# Drop rows with negative revenue, which are invalid values
valid = sessions[sessions["revenue"] >= 0]

# Average the daily conversion rates, so a single unusual day can't skew the result
daily = valid.groupby(["channel_group", "date"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
ai_report = daily.groupby("channel_group", as_index=False).agg(
    sessions=("sessions", "sum"),
    transactions=("transactions", "sum"),
    revenue=("revenue", "sum"),
    conversion_rate=("conversion_rate", "mean"),
)
ai_report = ai_report.sort_values("revenue", ascending=False).reset_index(drop=True)
ai_report

## Do the totals match?

In [ ]:
# ✏️ Create revenue_gap and removed


## Are the rates calculated the way they're defined?

In [ ]:
# ✏️ Create rate_gap


## The report

In [ ]:
# ✏️ Create report


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    revenue_gap, removed = found.get("revenue_gap"), found.get("removed")
    if not pd.api.types.is_number(revenue_gap):
        results.append((False, "revenue_gap should be a number: ai_report's revenue total minus the sessions table's."))
    elif abs(abs(float(revenue_gap)) - 62637.66) > 0.05:
        results.append((False, f"revenue_gap is {float(revenue_gap):,.2f}. Subtract sessions[\"revenue\"].sum() from ai_report[\"revenue\"].sum(): it should be 62,637.66."))
    elif float(revenue_gap) < 0:
        results.append((False, "revenue_gap has the right size, but it's negative. Subtract the sessions table's total from ai_report's: the AI's report has more revenue."))
    elif not isinstance(removed, pd.DataFrame) or "revenue" not in removed.columns:
        results.append((False, "removed should be a DataFrame: the rows of the sessions table with negative revenue, which the AI's filter dropped."))
    elif len(removed) != 530 or abs(float(removed["revenue"].sum()) + 62637.66) > 0.05:
        results.append((False, f"The AI's filter dropped 530 rows with −$62,637.66 of revenue. The removed table has {len(removed)} rows. Keep the rows where revenue is below 0."))
    else:
        results.append((True, "The AI's report has $62,637.66 too much revenue: its filter dropped 530 refund rows, with no sessions and negative revenue."))

    rate_gap = found.get("rate_gap")
    if not isinstance(rate_gap, pd.Series):
        results.append((False, "rate_gap should be a Series, with the channel groups as its index."))
    elif "Email" not in rate_gap.index:
        results.append((False, "rate_gap's index should be the channel groups: set_index(\"channel_group\") on ai_report first."))
    elif abs(float(rate_gap["Email"]) - 0.0019) > 0.00005:
        results.append((False, f"Email's rate_gap is {float(rate_gap['Email']):.5f}, and it should be 0.0019: ai_report's conversion_rate minus its transactions / sessions."))
    else:
        results.append((True, "The AI's rates aren't transactions divided by sessions: Email's is 0.0019 too high, because it's a mean of daily rates."))

    report = found.get("report")
    columns = ["channel_group", "sessions", "transactions", "revenue", "conversion_rate"]
    if not isinstance(report, pd.DataFrame) or [column for column in columns if column not in report.columns]:
        results.append((False, f"report should be a DataFrame with the columns {', '.join(columns)}."))
    else:
        revenue = float(report["revenue"].sum())
        pooled = report["transactions"] / report["sessions"]
        if len(report) != 9 or int(report["sessions"].sum()) != 255215:
            results.append((False, f"Q3 2024 has 9 channel groups and 255,215 sessions. The report table has {len(report)} rows and {int(report['sessions'].sum()):,} sessions."))
        elif abs(revenue - 517517.55) < 0.05:
            results.append((False, "The report's revenue is $517,517.55: the refund rows are still dropped. Group the whole sessions table."))
        elif abs(revenue - 454879.89) > 0.05:
            results.append((False, f"Q3 2024's revenue is $454,879.89. The report table has ${revenue:,.2f}."))
        elif ((report["conversion_rate"] - pooled).abs() > 0.00001).any():
            results.append((False, "The report's conversion rates are still a mean of daily rates. Divide transactions by sessions after grouping."))
        elif report.iloc[0]["channel_group"] != "Organic Search":
            results.append((False, "Sort the report table by revenue, highest first: Organic Search comes first."))
        else:
            results.append((True, "The report has 9 channel groups, with 255,215 sessions and $454,879.89 of revenue. Email converts best, at 5.35%."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
revenue_gap = ai_report["revenue"].sum() - sessions["revenue"].sum()
removed = sessions[sessions["revenue"] < 0]
print(revenue_gap)
removed[["sessions", "transactions", "revenue"]].sum()
```

```python
by_channel = ai_report.set_index("channel_group")
rate_gap = by_channel["conversion_rate"] - by_channel["transactions"] / by_channel["sessions"]
rate_gap.sort_values()
```

```python
report = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
report["conversion_rate"] = report["transactions"] / report["sessions"]
report = report.sort_values("revenue", ascending=False).reset_index(drop=True)
report
```

</details>